# AI-3 Session 1.1 — Query Refinement & Chunk Enrichment

Welcome to AI-3! In AI-2, you built a RAG pipeline. It works. Today we make it work **better**.

**What we're building today:**
- `hyde.py` — HyDE (Hypothetical Document Embeddings) for query-time improvement
- `enriched.py` — Question enrichment for ingestion-time improvement

Let's start by making sure your environment is ready.

In [ ]:
# Path setup — notebook is in notebooks/1_1/, pipeline is at repo root
import sys, os
from pathlib import Path

repo_root = str(Path().resolve().parent.parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)  # So .env and chroma_db paths resolve

from dotenv import load_dotenv
load_dotenv()
print(f"Working from: {os.getcwd()}")

In [ ]:
# Verify your environment
from pipeline.generation.generate import call_claude
from pipeline.retrieval.naive import naive_retrieve
from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection

print("All imports OK!")

In [ ]:
# Verify ChromaDB is loaded
collection = get_collection()
print(f"ChromaDB: {collection.count()} chunks loaded")

In [ ]:
# Verify API keys work (makes a real API call)
response = call_claude("Say hello in exactly 3 words.")
print(f"Claude says: {response}")

In [ ]:
# Verify Voyage AI connectivity
test_embedding = embed_texts(["test connectivity"])
print(f"Voyage AI: OK (embedding dim={len(test_embedding[0])})")

In [ ]:
# Verify Phoenix configuration
import os
from dotenv import load_dotenv
load_dotenv()

phoenix_key = os.getenv("PHOENIX_API_KEY")
phoenix_endpoint = os.getenv("PHOENIX_COLLECTOR_ENDPOINT")
phoenix_project = os.getenv("PHOENIX_PROJECT_NAME")

checks = []
if phoenix_key:
    checks.append("API Key: Set")
else:
    checks.append("API Key: MISSING -- go to app.phoenix.arize.com > Settings > API Keys")

if phoenix_endpoint:
    checks.append(f"Endpoint: {phoenix_endpoint}")
else:
    checks.append("Endpoint: MISSING")

if phoenix_project:
    checks.append(f"Project: {phoenix_project}")
else:
    checks.append("Project: MISSING -- set PHOENIX_PROJECT_NAME=ai3-yourname")

for c in checks:
    print(f"  Phoenix {c}")

if all([phoenix_key, phoenix_endpoint, phoenix_project]):
    print("\nPhoenix: All configured!")
else:
    print("\nPhoenix: Fix the MISSING items above in your .env file")

### Setup Complete

All five checks should pass:
1. Pipeline imports
2. ChromaDB loaded (173 chunks)
3. Claude API responding
4. Voyage AI returning embeddings
5. Phoenix configured

If anything is missing, fix it now before we continue.

## The Embedding Gap

Remember the PCA demo from AI-2 Session 3.2? Questions and answers cluster in **different neighborhoods** in embedding space.

- Questions encode interrogative structure: "What...", "How...", "When..."
- Answers encode declarative structure: "Employees receive...", "The policy states..."
- Same topic, **different embedding neighborhoods**

Let's see this in action with your current (naive) retrieval.

In [ ]:
# Run naive retrieval — observe what comes back
query = "What is the vacation policy?"
results = naive_retrieve(query, top_k=5)

print(f"Query: {query}\n")
for r in results:
    source = r["metadata"].get("source", "Unknown")
    score = r["score"]
    text_preview = r["text"][:120]
    print(f"  [{score:.3f}] {source}")
    print(f"           {text_preview}...\n")

In [ ]:
# Try a query where the gap is larger
query2 = "How does the company handle remote work requests?"
results2 = naive_retrieve(query2, top_k=5)

print(f"Query: {query2}\n")
for r in results2:
    source = r["metadata"].get("source", "Unknown")
    score = r["score"]
    text_preview = r["text"][:120]
    print(f"  [{score:.3f}] {source}")
    print(f"           {text_preview}...\n")

### What did you notice?

The results include chunks that *mention* the topic but aren't necessarily the **direct answers**. The query embedding lands near other question-like text, not near the answer text.

**Today's goal:** Close this gap with two techniques.
- **HyDE** — transform the query to sound like an answer (query time)
- **Question Enrichment** — transform the index to include questions (ingestion time)

## Solution 1: HyDE (Hypothetical Document Embeddings)

**The idea:** Before searching, ask Claude "What would a good answer to this question look like?" Then embed that hypothetical answer — not the original question — and use it for retrieval.

The hypothetical answer:
- Does NOT need to be correct
- Needs to SOUND like the kind of document that contains the real answer
- Lands in "answer space" near the real answers

```
NAIVE:    Question -> [embed] -> search (lands in question space)
HyDE:     Question -> [Claude] -> hypothetical answer -> [embed] -> search (lands in answer space)
```

### Let's build `generate_hypothetical_answer()`

This function takes a question and asks Claude to generate a hypothetical answer — a passage that sounds like it came from a real document.

In [ ]:
import anthropic

client = anthropic.Anthropic()


def generate_hypothetical_answer(question: str, domain: str = "company") -> str:
    """Generate a hypothetical answer for embedding-based retrieval.

    The answer does NOT need to be correct. It needs to SOUND like
    the kind of document that contains the real answer.

    Args:
        question: The user's original question
        domain: Hint for what kind of document to mimic

    Returns:
        A hypothetical answer passage (2-3 sentences)
    """
    system_prompt = f"""You are writing an excerpt from an internal {domain} policy document or handbook. 
    Write a brief passage (2-3 sentences) that answers the following question as if it were from such a document. 
    Make it sound like official, professional documentation. Do not include the question in your response."""
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=200,
        temperature=0.7,
        system=system_prompt,
        messages=[{"role": "user", "content": question}]
    )
    
    return response.content[0].text

In [ ]:
# Test it — what does the hypothetical answer look like?
test_question = "What is the vacation policy?"
hypothetical = generate_hypothetical_answer(test_question)
print(f"Question: {test_question}")
print(f"\nHypothetical answer:\n  {hypothetical}")

### Now let's build `hyde_retrieve()`

This function:
1. Generates a hypothetical answer
2. Embeds the hypothetical answer (NOT the question!)
3. Searches ChromaDB with that embedding
4. Returns results

In [ ]:
def hyde_retrieve(question: str, n_results: int = 5, domain: str = "company") -> list[dict]:
    """Retrieve using HyDE: embed a hypothetical answer instead of the question.

    Args:
        question: The user's original question
        n_results: Number of chunks to retrieve
        domain: Domain hint for hypothetical answer generation

    Returns:
        List of dicts with keys: text, metadata, score, hyde_answer
    """
    # Step 1: Generate hypothetical answer
    # Step 2: Embed the hypothetical answer (NOT the question!)
    # Step 3: Query ChromaDB with the hypothetical embedding
    # Step 4: Format results as list of dicts
    pass

In [ ]:
# Compare naive vs HyDE — validated example where HyDE clearly wins
query = "Who is the CEO and what are their priorities?"

# Naive
naive_results = naive_retrieve(query, top_k=3)
print("=== NAIVE TOP 3 ===")
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")
    print(f"           {r['text'][:100]}...\n")

# HyDE
hyde_results = hyde_retrieve(query, n_results=3)
print("=== HYPOTHETICAL ANSWER ===")
print(f"  {hyde_results[0]['hyde_answer']}\n")
print("=== HYDE TOP 3 ===")
for r in hyde_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')}")
    print(f"           {r['text'][:100]}...\n")

In [ ]:
# Another example — naive pulls an IRRELEVANT source here
query = "What are the performance review procedures?"

naive_results = naive_retrieve(query, top_k=3)
hyde_results = hyde_retrieve(query, n_results=3)

print(f"Query: {query}\n")
print("=== NAIVE TOP 3 ===")
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")
print()
print("=== HYDE TOP 3 ===")
for r in hyde_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")
print()
print("Notice: Naive's 3rd result is from expense_policy.md — completely irrelevant!")
print("HyDE keeps all results in employee_handbook.md where procedures actually live.")

### When does HyDE help? When doesn't it?

Try a **vague** query — HyDE amplifies whatever signal is in the question. No signal = no improvement.

Think about:
- Which queries show the biggest improvement?
- Which queries show no difference?
- What does the hypothetical answer look like for vague vs specific questions?

## Solution 2: Question Enrichment at Ingestion Time

**The idea:** Instead of transforming the query, transform the **index**. At ingestion time, ask Claude "What questions does this chunk answer?" Store those questions alongside the chunk.

At query time, the user's question matches against the *generated questions* — questions match questions, same neighborhood.

```
INGESTION:  Chunk -> [Claude: "what questions?"] -> questions
            -> [Embed chunk + questions] -> Store

QUERY TIME: Question -> [Embed] -> matches generated questions (free!)
```

Key difference from HyDE: **zero cost at query time**. You pay upfront at ingestion.

### Let's build `generate_questions_for_chunk()`

This function takes a chunk of text and asks Claude: "What questions does this text answer?"

In [ ]:
ENRICHED_COLLECTION = "northbrook_enriched"
N_QUESTIONS_PER_CHUNK = 3


def generate_questions_for_chunk(chunk_text: str, n_questions: int = N_QUESTIONS_PER_CHUNK) -> list[str]:
    """Generate questions that this chunk answers.
    
    Args:
        chunk_text: The text of the chunk
        n_questions: Number of questions to generate
    
    Returns:
        List of question strings
    """
    # Call Claude asking what questions this text answers.
    # Use temperature=0 for reproducibility.
    # Parse the response into a list of strings (one per line).
    pass

In [ ]:
# Get a sample chunk from ChromaDB and generate questions for it
sample = get_collection().get(limit=1, include=["documents", "metadatas"])
sample_text = sample["documents"][0]
sample_source = sample["metadatas"][0].get("source", "Unknown")

print(f"Source: {sample_source}")
print(f"Chunk: {sample_text[:200]}...\n")

questions = generate_questions_for_chunk(sample_text)
print("Generated questions:")
for q in questions:
    print(f"  - {q}")

### Now let's build `enrich_and_store()`

This function processes chunks, generates questions for each, creates combined embeddings, and stores them in a **new** ChromaDB collection (`northbrook_enriched`).

This is an **ingestion-time** operation. Run it once, and every query benefits.

In [ ]:
# Get chunks from the existing collection for enrichment
original = get_collection()
data = original.get(include=["documents", "metadatas"], limit=20)
chunks = [
    {"text": doc, "metadata": meta}
    for doc, meta in zip(data["documents"], data["metadatas"])
]
print(f"Got {len(chunks)} chunks for enrichment")

In [ ]:
import chromadb


def enrich_and_store(chunks: list[dict], collection_name: str = ENRICHED_COLLECTION) -> None:
    """Process chunks with enrichment and store in ChromaDB.
    
    For each chunk, generate N questions. Store EACH question as its own
    row in the vector store, with the chunk text as the 'documents' field.
    
    Result: the enriched collection has ~3x rows as chunks. Each row's
    embedding is a pure question embedding — questions match questions
    at query time.
    
    Args:
        chunks: List of dicts with keys: text, metadata
        collection_name: Name for the enriched collection
    """
    # Step 1: Get or create the enriched collection (delete if exists for clean ingest)
    # Step 2: For each chunk:
    #   a. Generate N questions
    #   b. Embed ALL questions in one call (batch)
    #   c. For each question, add a row:
    #      - id = unique per (source, chunk_idx, q_idx)
    #      - documents = chunk text (what we return on retrieval)
    #      - embeddings = the question's embedding (pure question-space)
    #      - metadata = original metadata + chunk_index + source_question
    pass

In [ ]:
# Enrich 20 chunks (takes ~30-60 seconds)
enrich_and_store(chunks)

### Finally, `enriched_retrieve()`

This function queries the enriched collection. It's structurally identical to naive retrieval — but the collection has enriched embeddings that include question context.

In [ ]:
def enriched_retrieve(question: str, n_results: int = 5, collection_name: str = ENRICHED_COLLECTION) -> list[dict]:
    """Retrieve from the enriched collection.
    
    Because each chunk has N rows (one per question), we over-fetch and
    dedup by chunk_index to return top-N unique chunks.
    
    Args:
        question: The user's question
        n_results: Number of UNIQUE chunks to return
        collection_name: The enriched collection name
    
    Returns:
        List of dicts with keys: text, metadata, score, matched_question
    """
    # Step 1: Embed the question
    # Step 2: Over-fetch n_results * N_QUESTIONS_PER_CHUNK rows
    # Step 3: Dedup by chunk_index — keep best score per chunk
    # Step 4: Sort by score descending and return top-N unique
    pass

In [ ]:
# Three-way comparison — enrichment wins here, HyDE actually fails!
query = "What happened at the Q4 board meeting?"

print(f"Query: {query}\n")

# Naive
naive_results = naive_retrieve(query, top_k=3)
print("=== NAIVE ===")
for r in naive_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")

# HyDE
hyde_results = hyde_retrieve(query, n_results=3)
print("\n=== HYDE ===")
print(f"  Hypothetical: {hyde_results[0]['hyde_answer'][:100]}...")
for r in hyde_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")

# Enriched
enriched_results = enriched_retrieve(query, n_results=3)
print("\n=== ENRICHED ===")
for r in enriched_results:
    print(f"  [{r['score']:.3f}] {r['metadata'].get('source', '?')} — {r['text'][:80]}...")

print("\nNotice: HyDE may MISS the Q4 board meeting doc (hypothetical hallucinated).")
print("Enrichment retrieves the right docs because generated questions match directly.")

## Save to Python Modules

Your notebook code works. Now let's make it a proper module that your app can import.

### `pipeline/retrieval/hyde.py`

Copy your working functions into this file. The file already exists with imports set up — add your function implementations.

```python
"""
hyde.py -- Hypothetical Document Embeddings

Transform queries by generating hypothetical answers, then embed the
hypothetical answer for retrieval instead of the raw question.
"""

import anthropic
from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection

client = anthropic.Anthropic()


def generate_hypothetical_answer(question: str, domain: str = "company") -> str:
    # Your implementation from the notebook
    ...

def hyde_retrieve(question: str, n_results: int = 5, domain: str = "company") -> list[dict]:
    # Your implementation from the notebook
    ...
```

### `pipeline/retrieval/enriched.py`

```python
"""
enriched.py -- Question Enrichment at Ingestion Time

At ingestion time, generate questions each chunk answers.
Embed those questions alongside the chunk for better query matching.
"""

import anthropic
from pipeline.embeddings.embed import embed_texts
from pipeline.ingestion.store import get_collection

client = anthropic.Anthropic()

ENRICHED_COLLECTION = "northbrook_enriched"


def generate_questions_for_chunk(chunk_text: str, n_questions: int = 3) -> list[str]:
    # Your implementation from the notebook
    ...

def enrich_and_store(chunks: list[dict], collection_name: str = ENRICHED_COLLECTION) -> None:
    # Your implementation from the notebook
    ...

def enriched_retrieve(question: str, n_results: int = 5, collection_name: str = ENRICHED_COLLECTION) -> list[dict]:
    # Your implementation from the notebook
    ...
```

In [ ]:
# After saving your .py files, verify they import correctly
# (restart kernel first if needed)
from pipeline.retrieval.hyde import hyde_retrieve
from pipeline.retrieval.enriched import enriched_retrieve

print("Module imports OK!")

## What You Built Today

- **`hyde.py`** — transforms queries into answer space (online, per-query cost)
- **`enriched.py`** — transforms index into question space (offline, per-chunk cost)
- Both close the embedding gap you saw in AI-2's PCA demo
- Both improve retrieval over the naive baseline

## Next Session — 1.2: Proving Improvement

You improved retrieval. But "I can see it looks better" is not engineering. Next session you build the evidence:

- **Golden test sets** — curated question/answer pairs
- **Before/after evaluation** — quantify the improvement
- **Regression framework** — catch when changes break things

**Questions to think about:**
1. How do you **prove** retrieval is better, not just feel better?
2. If you change the pipeline tomorrow, how do you know you didn't break what was working?
3. Can you measure the embedding gap closing?

**Make sure both `hyde.py` and `enriched.py` are working before Session 1.2.**